# Simplified Test: 200 Balanced Samples with Unified Module

This notebook demonstrates the simplified API using the refactored modules.

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, '/shared_data0/weiqiuy/llm_cholec_organ/src')

from endopoint.datasets import build_dataset
from endopoint.fewshot import UnifiedFewShotSelector

print("✅ Modules imported")

## Test Function Using New Pipeline Method

In [ ]:
def test_dataset_simplified(dataset_name, dataset_config=None, force_regenerate=False):
    """Simplified test using the new pipeline method.
    
    All the balance analysis and configuration is now handled by the module.
    
    Args:
        dataset_name: Name of the dataset
        dataset_config: Optional dataset configuration
        force_regenerate: If True, delete cache and regenerate
    """
    print("\n" + "="*70)
    print(f"Testing: {dataset_name}")
    print("="*70)
    
    # Default configurations
    if dataset_config is None:
        configs = {
            "cholecseg8k_local": {
                "data_dir": "/shared_data0/weiqiuy/datasets/cholecseg8k"
            },
            "cholec_organs": {
                "data_dir": "/shared_data0/weiqiuy/real_drs/data/abdomen_exlib",
                "video_globs": "public",
                "gen_seed": 56,
                "train_val_seed": 0
            },
            "cholec_gonogo": {
                "data_dir": "/shared_data0/weiqiuy/real_drs/data/abdomen_exlib",
                "video_globs": "public",
                "gen_seed": 56,
                "train_val_seed": 0
            }
        }
        dataset_config = configs.get(dataset_name, {})
    
    # Load dataset
    dataset = build_dataset(dataset_name, **dataset_config)
    
    print(f"\n📊 Dataset Info:")
    print(f"  Tag: {dataset.dataset_tag}")
    print(f"  Train: {dataset.total('train')} examples")
    print(f"  Classes: {len(dataset.label_ids)}")
    
    # Create selector with 200 samples and 30% minimum quota
    output_dir = Path(f"/shared_data0/weiqiuy/llm_cholec_organ/data_info/{dataset_name}_balanced_200")
    
    # If force_regenerate, clear the cached balanced indices
    if force_regenerate:
        cache_file = output_dir / "balanced_test_indices_advanced_200.json"
        if cache_file.exists():
            print(f"⚠️ Removing old cache file: {cache_file.name}")
            cache_file.unlink()
            print("  Cache cleared - will regenerate with correct parameters")
    
    selector = UnifiedFewShotSelector(
        dataset=dataset,
        output_dir=output_dir,
        n_test_samples=200,  # 200 balanced samples
        n_pos_examples=1,
        n_neg_absent=1,
        n_neg_wrong=1,
        min_pixels=50,
        seed=42,
        cache_enabled=True
    )
    
    print(f"📁 Output directory: {output_dir}")
    print(f"\n🔧 Configuration:")
    print(f"  Test samples: 200")
    print(f"  Min quota for rare classes: 30% (60 samples)")
    print(f"  Cap for abundant classes: 70% of ideal distribution")
    
    # Run the complete pipeline with all analysis
    results = selector.run_balanced_selection_pipeline(
        split="train",
        visualize=True,  # Print detailed analysis
        save_summary=True  # Save summary to file
    )
    
    # Verify we got 200 samples
    actual_samples = len(results['test_indices'])
    if actual_samples != 200:
        print(f"\n⚠️ WARNING: Expected 200 samples but got {actual_samples}")
        print("  Consider running with force_regenerate=True to clear cache")
    
    # The pipeline already printed everything, just return results
    return results

print("✅ Simplified test function updated with 30% minimum quota (60 samples)")

## Test All Datasets

In [ ]:
# Test CholecSeg8k - force regeneration to use correct 20% quota (40 samples)
results_cholecseg8k = test_dataset_simplified("cholecseg8k_local", force_regenerate=True)

In [ ]:
# Test CholecOrgans - force regeneration to use correct 20% quota  
results_organs = test_dataset_simplified("cholec_organs", force_regenerate=True)

In [ ]:
# Test CholecGoNoGo - force regeneration to use correct 20% quota
results_gonogo = test_dataset_simplified("cholec_gonogo", force_regenerate=True)

## Summary Comparison

In [ ]:
# Compare results across datasets
print("\n" + "="*70)
print("📊 Summary: 200 Balanced Test Samples per Dataset")
print("="*70)

all_results = [
    ("cholecseg8k_local", results_cholecseg8k),
    ("cholec_organs", results_organs),
    ("cholec_gonogo", results_gonogo)
]

print(f"\n{'Dataset':<20} {'Classes':<10} {'Train Size':<12} {'Test Size':<12} {'Balance Improvement'}")
print("-" * 80)

for dataset_name, res in all_results:
    metrics = res['balance_comparison']['metrics']
    improvement = metrics['balance_improvement_pct']
    
    # Color code based on improvement
    if improvement > 20:
        color = "🟢"
    elif improvement > 10:
        color = "🟡"
    elif improvement > 0:
        color = "🟠"
    else:
        color = "🔴"
    
    print(f"{dataset_name:<20} {len(res['test_indices']):<10} "
          f"{res['presence_matrix_shape'][0]:<12} {len(res['test_indices']):<12} "
          f"{color} {improvement:+6.1f}%")

print("\n✅ All datasets now have 200 balanced test samples!")
print("📁 Summaries saved in each dataset's output directory as 'pipeline_summary.json'")

## Smart Few-Shot Selection for Combined Mode

This section implements smart selection of few-shot examples for combined mode evaluation.
The goal is to select the minimum number of training images that collectively cover all organs,
where each selected image will include bounding boxes for all organs present in that image.

In [ ]:
import json
import numpy as np
from typing import Dict, List, Set, Tuple
from collections import defaultdict

def load_organ_specific_fewshot_plan(plan_file: Path) -> Dict:
    """Load the organ-specific few-shot plan."""
    with open(plan_file, 'r') as f:
        return json.load(f)

def analyze_image_coverage(fewshot_plan: Dict, presence_matrix: np.ndarray, label_ids: Dict[str, int]) -> Dict:
    """Analyze which organs are covered by each training image in the few-shot plan.
    
    Returns:
        Dict with image indices as keys and sets of covered organs as values
    """
    image_coverage = defaultdict(set)
    
    # Go through each organ's positive examples
    for organ_name, examples in fewshot_plan.items():
        if not isinstance(examples, dict) or 'positive' not in examples:
            continue
            
        organ_id = label_ids.get(organ_name)
        if organ_id is None:
            continue
            
        # Each positive example shows this organ
        for example in examples.get('positive', []):
            idx = example['idx']
            image_coverage[idx].add(organ_name)
    
    # Also check what other organs are present in these images
    for idx in image_coverage.keys():
        for organ_name, organ_id in label_ids.items():
            if presence_matrix[idx, organ_id] == 1:
                image_coverage[idx].add(organ_name)
    
    return dict(image_coverage)

def greedy_set_cover(image_coverage: Dict[int, Set[str]], target_organs: Set[str]) -> List[int]:
    """Find minimum set of images that cover all target organs using greedy algorithm.
    
    Args:
        image_coverage: Dict mapping image index to set of organs present
        target_organs: Set of organ names to cover
        
    Returns:
        List of selected image indices
    """
    selected_images = []
    covered_organs = set()
    remaining_organs = target_organs.copy()
    
    # Make a copy of coverage to modify
    available_coverage = image_coverage.copy()
    
    while remaining_organs:
        # Find image that covers the most uncovered organs
        best_image = None
        best_new_organs = set()
        best_count = 0
        
        for idx, organs in available_coverage.items():
            new_organs = organs & remaining_organs
            if len(new_organs) > best_count:
                best_image = idx
                best_new_organs = new_organs
                best_count = len(new_organs)
        
        if best_image is None:
            print(f"Warning: Could not cover all organs. Missing: {remaining_organs}")
            break
            
        # Add this image
        selected_images.append(best_image)
        covered_organs.update(best_new_organs)
        remaining_organs -= best_new_organs
        
        # Remove selected image from available pool
        del available_coverage[best_image]
    
    return selected_images

print("✅ Smart selection functions defined")

In [ ]:
# Load CholecSeg8k dataset and its few-shot plan
dataset_name = "cholecseg8k_local"
data_dir = Path(f"/shared_data0/weiqiuy/llm_cholec_organ/data_info/{dataset_name}_balanced_200")

# Load the organ-specific bbox few-shot plan
bbox_plan_file = data_dir / "fewshot_plan_bbox_200.json"
if not bbox_plan_file.exists():
    print(f"❌ Bbox few-shot plan not found: {bbox_plan_file}")
else:
    fewshot_plan = load_organ_specific_fewshot_plan(bbox_plan_file)
    print(f"✅ Loaded few-shot plan from {bbox_plan_file.name}")
    
    # Show organ distribution in the plan
    print("\n📊 Organs in few-shot plan:")
    for organ_name, examples in fewshot_plan.items():
        if isinstance(examples, dict) and 'positive' in examples:
            n_pos = len(examples.get('positive', []))
            n_neg_absent = len(examples.get('negative_absent', []))
            n_neg_wrong = len(examples.get('negative_wrong', []))
            print(f"  {organ_name:30} Pos: {n_pos}, Neg-absent: {n_neg_absent}, Neg-wrong: {n_neg_wrong}")

In [ ]:
# Load dataset and presence matrix
from endopoint.datasets import build_dataset

dataset = build_dataset(dataset_name, data_dir="/shared_data0/weiqiuy/datasets/cholecseg8k")

# Load presence matrix
presence_matrix_file = data_dir / "presence_matrix_train.npy"
presence_matrix = np.load(presence_matrix_file)
print(f"✅ Loaded presence matrix: {presence_matrix.shape}")

# Get label IDs mapping
label_ids = dataset.label_ids
label_names = dataset.label_names
print(f"✅ Dataset has {len(label_ids)} organs")

# Analyze image coverage from the few-shot plan
image_coverage = analyze_image_coverage(fewshot_plan, presence_matrix, label_ids)
print(f"\n📊 Image coverage analysis:")
print(f"  Total unique images in few-shot plan: {len(image_coverage)}")

# Show top images by organ coverage
sorted_coverage = sorted(image_coverage.items(), key=lambda x: len(x[1]), reverse=True)
print(f"\n  Top 5 images by organ coverage:")
for idx, organs in sorted_coverage[:5]:
    print(f"    Image {idx:5d}: {len(organs)} organs - {', '.join(sorted(organs))}")

In [ ]:
# Run greedy set cover to find minimum images covering all organs
target_organs = set(label_names)
selected_indices = greedy_set_cover(image_coverage, target_organs)

print(f"\n🎯 Smart Selection Results:")
print(f"  Target: Cover all {len(target_organs)} organs")
print(f"  Selected: {len(selected_indices)} images (vs {len(image_coverage)} total available)")
print(f"  Reduction: {(1 - len(selected_indices)/len(image_coverage))*100:.1f}%")

# Verify coverage
covered = set()
for idx in selected_indices:
    covered.update(image_coverage[idx])
print(f"\n✅ Verification: {len(covered)} organs covered")

if covered == target_organs:
    print("  All organs successfully covered!")
else:
    missing = target_organs - covered
    print(f"  ⚠️ Missing organs: {missing}")

# Show selected images
print(f"\n📋 Selected images and their organs:")
for i, idx in enumerate(selected_indices, 1):
    organs = sorted(image_coverage[idx])
    print(f"  {i:2d}. Image {idx:5d}: {len(organs)} organs - {', '.join(organs)}")

In [ ]:
# Create combined few-shot examples with all bounding boxes
from endopoint.datasets.cholecseg8k_local import CholecSeg8kLocalAdapter

# Initialize dataset adapter
adapter = CholecSeg8kLocalAdapter(data_dir="/shared_data0/weiqiuy/datasets/cholecseg8k")

# Create combined examples
combined_examples = []

for idx in selected_indices:
    # Get the example from dataset
    example = adapter.get_example('train', idx)
    
    # Create combined bounding boxes for all present organs
    combined_bboxes = {}
    
    for organ_name in image_coverage[idx]:
        organ_id = label_ids[organ_name]
        
        # Get bounding box if organ is present
        if presence_matrix[idx, organ_id] == 1:
            bbox = adapter.get_bounding_box(example, organ_id)
            if bbox is not None:
                combined_bboxes[organ_name] = {
                    "bbox": bbox,
                    "present": True
                }
    
    # Add to combined examples
    combined_examples.append({
        "idx": idx,
        "image_path": f"train_{idx:05d}.jpg",  # Placeholder path
        "organs": list(image_coverage[idx]),
        "bboxes": combined_bboxes
    })

print(f"✅ Created {len(combined_examples)} combined few-shot examples")

# Show example structure
if combined_examples:
    print(f"\n📦 Example structure (first example):")
    first = combined_examples[0]
    print(f"  Index: {first['idx']}")
    print(f"  Organs present: {len(first['organs'])}")
    print(f"  Bounding boxes: {len(first['bboxes'])}")
    
    # Show first few bboxes
    for i, (organ, bbox_info) in enumerate(list(first['bboxes'].items())[:3]):
        bbox = bbox_info['bbox']
        print(f"    {organ}: [{bbox[0]:.1f}, {bbox[1]:.1f}, {bbox[2]:.1f}, {bbox[3]:.1f}]")

In [ ]:
# Save the combined few-shot plan
output_file = data_dir / "fewshot_plan_bbox_combined_smart.json"

# Format for saving
save_data = {
    "metadata": {
        "creation_method": "smart_set_cover",
        "total_organs": len(target_organs),
        "num_images": len(combined_examples),
        "reduction_ratio": f"{(1 - len(selected_indices)/len(image_coverage))*100:.1f}%",
        "source_plan": "fewshot_plan_bbox_200.json"
    },
    "examples": combined_examples
}

# Save to file
with open(output_file, 'w') as f:
    json.dump(save_data, f, indent=2)

print(f"\n💾 Saved combined few-shot plan to: {output_file.name}")
print(f"   Total examples: {len(combined_examples)}")
print(f"   File size: {output_file.stat().st_size / 1024:.1f} KB")

# Summary statistics
total_bboxes = sum(len(ex['bboxes']) for ex in combined_examples)
avg_organs_per_image = total_bboxes / len(combined_examples)

print(f"\n📊 Summary Statistics:")
print(f"   Average organs per image: {avg_organs_per_image:.1f}")
print(f"   Total bounding boxes: {total_bboxes}")
print(f"   Efficiency gain: {len(image_coverage)/len(combined_examples):.1f}x fewer images needed")